In [14]:
# import yfinance as yf
# import pandas as pd
# import numpy as np
# try:
#     import talib
# except ImportError:
#     talib = None

# # Custom RSI function if ta-lib is unavailable
# def MA_RSI(ticker):
#   def calculate_rsi(data, periods=14):
#       delta = data.diff()
#       gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
#       loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
#       rs = gain / loss
#       return 100 - (100 / (1 + rs))

#   # Download XAU/USD data

#   start_date = "2024-01-01"
#   end_date = "2024-12-31"
#   data = yf.download(ticker, start=start_date, end=end_date, interval="1h")
#   data = data.droplevel(1,axis =1)
#   # Calculate indicators
#   data['SMA200'] = data['Close'].rolling(window=200).mean()
#   if talib:
#       data['RSI'] = talib.RSI(data['Close'], timeperiod=14)
#   else:
#       data['RSI'] = calculate_rsi(data['Close'], periods=14)
#   data['SwingLow'] = data['Low'].rolling(window=5).min()

#   # Initialize variables
#   account = 10000  # Starting balance
#   risk_per_trade = 0.01  # 1% risk
#   trades = []
#   position = None
#   data = data.dropna()  # Remove rows with NaN

#   # Backtest loop
#   for i in range(1, len(data)):
#       # Trend filter: Price > SMA200
#       if data['Close'].iloc[i] > data['SMA200'].iloc[i]:
#           # Entry: RSI crosses above 50 and RSI < 75
#           if (data['RSI'].iloc[i-1] <= 50 and data['RSI'].iloc[i] > 50 and
#               data['RSI'].iloc[i] < 75 and position is None):
#               entry_price = data['Close'].iloc[i]
#               stop_loss = data['SwingLow'].iloc[i]
#               risk = entry_price - stop_loss
#               if risk <= 0:  # Skip invalid stop-loss
#                   continue
#               take_profit = entry_price + 2 * risk  # 2:1 RRR
#               position_size = (account * risk_per_trade) / risk  # Units to risk 1%
#               position = {'entry': entry_price, 'stop_loss': stop_loss,
#                           'take_profit': take_profit, 'size': position_size, 'index': i}

#           # Exit logic
#           if position:
#               # Check stop-loss or take-profit
#               if data['Low'].iloc[i] <= position['stop_loss']:
#                   exit_price = position['stop_loss']
#                   profit = (exit_price - position['entry']) * position['size'] - 0.2  # Spread
#                   trades.append({
#                       'entry_date': data.index[position['index']],
#                       'exit_date': data.index[i],
#                       'entry_price': position['entry'],
#                       'exit_price': exit_price,
#                       'profit': profit
#                   })
#                   account += profit
#                   position = None
#               elif data['High'].iloc[i] >= position['take_profit']:
#                   exit_price = position['take_profit']
#                   profit = (exit_price - position['entry']) * position['size'] - 0.2
#                   trades.append({
#                       'entry_date': data.index[position['index']],
#                       'exit_date': data.index[i],
#                       'entry_price': position['entry'],
#                       'exit_price': exit_price,
#                       'profit': profit
#                   })
#                   account += profit
#                   position = None

#   # Performance metrics
#   trades_df = pd.DataFrame(trades)
#   print(ticker)
#   if not trades_df.empty:
#       win_rate = len(trades_df[trades_df['profit'] > 0]) / len(trades_df)
#       total_profit = trades_df['profit'].sum()
#       avg_profit = trades_df['profit'].mean()
#       sharpe_ratio = trades_df['profit'].mean() / trades_df['profit'].std() * np.sqrt(252) if trades_df['profit'].std() != 0 else 0
#       equity = [10000] + (trades_df['profit'].cumsum() + 10000).tolist()
#       max_drawdown = max([1 - e / max(equity[:i+1]) for i, e in enumerate(equity[1:], 1)]) if equity else 0

#       # print(f"Number of Trades: {len(trades)}")
#       # print(f"Win Rate: {win_rate:.2%}")
#       # print(f"Total Profit: ${total_profit:.2f}")
#       # print(f"Average Profit per Trade: ${avg_profit:.2f}")
#       # print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
#       # print(f"Max Drawdown: {max_drawdown:.2%}")
#       return sharpe_ratio, total_profit, avg_profit, win_rate, max_drawdown
#   else:
#       print("No trades executed.")
#       return None




In [15]:
import yfinance as yf
import pandas as pd
import numpy as np
try:
    import talib
except ImportError:
    talib = None

def MA_RSI(ticker):
    # Custom RSI function if ta-lib is unavailable
    def calculate_rsi(data, periods=14):
        delta = data.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))

    # Download data for the specified ticker
    start_date = "2020-06-01"
    end_date = "2025-07-15"
    data = yf.download(ticker, start=start_date, end=end_date, interval="1d")

    # Handle multi-level columns if present
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel(1, axis=1)

    # Calculate indicators
    data['SMA200'] = data['Close'].rolling(window=200).mean()
    if talib:
        data['RSI'] = talib.RSI(data['Close'], timeperiod=14)
    else:
        data['RSI'] = calculate_rsi(data['Close'], periods=14)
    data['SwingLow'] = data['Low'].rolling(window=5).min()

    # Initialize variables
    account = 10000  # Starting balance
    risk_per_trade = 0.01  # 1% risk
    trades = []
    position = None
    data = data.dropna()  # Remove rows with NaN

    # Backtest loop
    for i in range(1, len(data)):
        # Trend filter: Price > SMA200
        if data['Close'].iloc[i] > data['SMA200'].iloc[i]:
            # Entry: RSI crosses above 50 and RSI < 75
            if (data['RSI'].iloc[i-1] <= 50 and data['RSI'].iloc[i] > 50 and
                data['RSI'].iloc[i] < 75 and position is None):
                entry_price = data['Close'].iloc[i]
                stop_loss = data['SwingLow'].iloc[i]
                risk = entry_price - stop_loss
                if risk <= 0:  # Skip invalid stop-loss
                    continue
                take_profit = entry_price + 2 * risk  # 2:1 RRR
                position_size = (account * risk_per_trade) / risk  # Units to risk 1%
                position = {
                    'entry': entry_price,
                    'stop_loss': stop_loss,
                    'take_profit': take_profit,
                    'size': position_size,
                    'index': i,
                    'entry_date': data.index[i]
                }

            # Exit logic
            if position:
                # Check stop-loss or take-profit
                if data['Low'].iloc[i] <= position['stop_loss']:
                    exit_price = position['stop_loss']
                    profit = (exit_price - position['entry']) * position['size'] - 0.2  # Spread
                    holding_period_hours = (data.index[i] - position['entry_date']).total_seconds() / 3600
                    trades.append({
                        'entry_date': position['entry_date'],
                        'exit_date': data.index[i],
                        'entry_price': position['entry'],
                        'exit_price': exit_price,
                        'profit': profit,
                        'holding_period_hours': holding_period_hours
                    })
                    account += profit
                    position = None
                elif data['High'].iloc[i] >= position['take_profit']:
                    exit_price = position['take_profit']
                    profit = (exit_price - position['entry']) * position['size'] - 0.2
                    holding_period_hours = (data.index[i] - position['entry_date']).total_seconds() / 3600
                    trades.append({
                        'entry_date': position['entry_date'],
                        'exit_date': data.index[i],
                        'entry_price': position['entry'],
                        'exit_price': exit_price,
                        'profit': profit,
                        'holding_period_hours': holding_period_hours
                    })
                    account += profit
                    position = None

    # Performance metrics
    trades_df = pd.DataFrame(trades)
    print(ticker)
    if not trades_df.empty:
        win_rate = len(trades_df[trades_df['profit'] > 0]) / len(trades_df)
        total_profit = trades_df['profit'].sum()
        avg_profit = trades_df['profit'].mean()
        sharpe_ratio = trades_df['profit'].mean() / trades_df['profit'].std() * np.sqrt(252) if trades_df['profit'].std() != 0 else 0
        equity = [10000] + (trades_df['profit'].cumsum() + 10000).tolist()
        max_drawdown = max([1 - e / max(equity[:i+1]) for i, e in enumerate(equity[1:], 1)]) if equity else 0
        avg_holding_time = trades_df['holding_period_hours'].mean()
        return sharpe_ratio, total_profit, avg_profit, win_rate, max_drawdown , avg_holding_time
        # return {
        #     'sharpe_ratio': sharpe_ratio,
        #     'total_profit': total_profit,
        #     'avg_profit': avg_profit,
        #     'win_rate': win_rate,
        #     'max_drawdown': max_drawdown,
        #     'avg_holding_time_hours': avg_holding_time
        # }
    else:
        print("No trades executed.")
        return None

In [16]:
tickers = [
    # Forex Symbols
    "EURUSD=X",  # Euro / US Dollar
    "GBPUSD=X",  # British Pound / US Dollar
    "USDJPY=X",  # US Dollar / Japanese Yen
    "AUDUSD=X",  # Australian Dollar / US Dollar
    "USDCAD=X",  # US Dollar / Canadian Dollar
    "USDCHF=X",  # US Dollar / Swiss Franc
    "NZDUSD=X",  # New Zealand Dollar / US Dollar
    "EURGBP=X",  # Euro / British Pound
    "EURJPY=X",  # Euro / Japanese Yen
    "GBPJPY=X",  # British Pound / Japanese Yen

    # Commodities Symbols
    "GC=F",      # Gold Futures
    "SI=F",      # Silver Futures
    "CL=F",      # Crude Oil Futures
    "NG=F",      # Natural Gas Futures
    "ZC=F",      # Corn Futures
    "ZS=F",      # Soybean Futures
    "ZW=F",      # Wheat Futures
    "HG=F",      # Copper Futures
    "PL=F",      # Platinum Futures
    "PA=F",       # Palladium Futures
    "BTC-USD",   # Bitcoin
    "ETH-USD",
    "XRP-USD",
    "SPY",
    "QQQ",
    "KO",
    "AAPL",
    "MSFT",
    "AMZN",
    "GOOGL",
    "TSLA"
]
asset_performance = []
for ticker in tickers:
  sharpe_ratio, total_profit, avg_profit, win_rate, max_drawdown, avg_holding_time = MA_RSI(ticker)
  ticker_performance = {
      "Ticker": ticker,
      "sharpe_ratio": sharpe_ratio,
      "total_profit": total_profit,
      "avg_profit": avg_profit,
      "win_rate": win_rate,
      "max_drawdown": max_drawdown,
      "average_holding_time": avg_holding_time
  }
  asset_performance.append(ticker_performance)
  print(f"Ticker: {ticker}")
  print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
  print(f"Total Profit: ${total_profit:.2f}")
  print(f"Average Profit per Trade: ${avg_profit:.2f}")
  print(f"Win Rate: {win_rate:.2%}")
  print(f"Max Drawdown: {max_drawdown:.2%}")
  print(f"Average Holding time: {avg_holding_time}")
  print("\n")


/tmp/ipython-input-15-1855785468.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, interval="1d")
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-15-1855785468.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, interval="1d")


EURUSD=X
Ticker: EURUSD=X
Sharpe Ratio: 6.16
Total Profit: $811.94
Average Profit per Trade: $62.46
Win Rate: 53.85%
Max Drawdown: 1.99%
Average Holding time: 1728.0




[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-15-1855785468.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, interval="1d")
[*********************100%***********************]  1 of 1 completed


GBPUSD=X
Ticker: GBPUSD=X
Sharpe Ratio: -1.14
Total Profit: $-224.09
Average Profit per Trade: $-9.74
Win Rate: 30.43%
Max Drawdown: 7.81%
Average Holding time: 893.2173913043479


USDJPY=X
Ticker: USDJPY=X
Sharpe Ratio: 4.35
Total Profit: $909.69
Average Profit per Trade: $43.32
Win Rate: 47.62%
Max Drawdown: 1.99%
Average Holding time: 493.7142857142857




/tmp/ipython-input-15-1855785468.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, interval="1d")
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-15-1855785468.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, interval="1d")
[*********************100%***********************]  1 of 1 completed


...

MSFT
Ticker: MSFT
Sharpe Ratio: 3.67
Total Profit: $800.38
Average Profit per Trade: $36.38
Win Rate: 45.45%
Max Drawdown: 4.91%
Average Holding time: 1021.0909090909091


AMZN
Ticker: AMZN
Sharpe Ratio: 6.84
Total Profit: $1244.35
Average Profit per Trade: $69.13
Win Rate: 55.56%
Max Drawdown: 2.98%
Average Holding time: 908.0




/tmp/ipython-input-15-1855785468.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, interval="1d")
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-15-1855785468.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, interval="1d")


GOOGL
Ticker: GOOGL
Sharpe Ratio: 4.97
Total Profit: $1020.23
Average Profit per Trade: $51.01
Win Rate: 50.00%
Max Drawdown: 1.99%
Average Holding time: 1106.4




[*********************100%***********************]  1 of 1 completed

TSLA
Ticker: TSLA
Sharpe Ratio: -2.05
Total Profit: $-315.01
Average Profit per Trade: $-17.50
Win Rate: 27.78%
Max Drawdown: 4.91%
Average Holding time: 794.6666666666666




In [17]:
strat_profit = 0
for asset in asset_performance:
  profit = asset['total_profit']
  strat_profit += profit
print(strat_profit)

15031.504926719743


In [18]:
# prompt: sort assset according to profit

import pandas as pd
asset_performance_df = pd.DataFrame(asset_performance)
sorted_assets = asset_performance_df.sort_values(by='total_profit', ascending=False)

print("\nAssets sorted by Total Profit:")
sorted_assets


Assets sorted by Total Profit:


,Ticker,sharpe_ratio,total_profit,avg_profit,win_rate,max_drawdown,average_holding_time
13,NG=F,9.434449,2170.666839,98.666675,0.636364,0.019932,576.000000
10,GC=F,7.179818,1923.351734,71.235249,0.555556,0.049785,509.333333
11,SI=F,7.826812,1468.947458,77.313024,0.578947,0.039826,760.421053
26,AAPL,6.803089,1354.047216,67.702361,0.550000,0.039480,1047.600000
28,AMZN,6.844338,1244.354286,69.130794,0.555556,0.029755,908.000000
25,KO,7.305349,1027.977487,73.426963,0.571429,0.019938,1858.285714
29,GOOGL,4.971230,1020.230944,51.011547,0.500000,0.019938,1106.400000
2,USDJPY=X,4.354484,909.685851,43.318374,0.476190,0.019939,493.714286
0,EURUSD=X,6.164445,811.944707,62.457285,0.538462,0.019938,1728.000000
27,MSFT,3.673612,800.384483,36.381113,0.454545,0.049103,1021.090909
